# **Notebook 1: Data Cleaning & Order-Level Dataset Construction**
## Capstone: Customer Satisfaction Analysis for the Olist Brazilian E-Commerce Marketplace
---

### 📋 TO-DO: Before Running This Notebook

**What you NEED for this notebook:**
- [ ] A Kaggle account + API token (`kaggle.json`) so `kagglehub` can download the dataset
- [ ] Internet access (to download `olistbr/brazilian-ecommerce` from Kaggle)

**What this notebook will CREATE:**
- [ ] `olist_clean.csv` — a cleaned, **order-level** dataset (~96k rows) with the binary target `satisfied`.
  _(Required by: Notebook 2 — EDA & Baseline Model)_

> **Stage 1 scope (very important):** This notebook does **structural** cleaning only — merging the nine
> tables, restricting to delivered+reviewed orders, fixing types, and creating the target.
> We do **NOT** scale, impute, encode, or build statistical features (e.g., `seller_avg_score`) here.
> Those happen **after** the train/test split in Notebook 2 to prevent data leakage.
---

## Setup — Install & Import

**Hints & Tips:**
* `kagglehub` downloads the dataset to a local cache and returns the folder path.
* If you prefer, you can skip `kagglehub` and upload the 9 CSVs manually — just set `DATA_DIR` to that folder.

In [1]:
# Colab already ships pandas/numpy/matplotlib/seaborn — we only need kagglehub here.
# (Avoid `-U` on numpy/pandas in Colab: upgrading them mid-session breaks the kernel.)
!pip install -q kagglehub

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

In [2]:
import os, glob
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

RANDOM_STATE = 42  # used consistently across all notebooks for reproducibility

## **Stage 1: Data Understanding & Readiness**
### **Task 1.3: Load, Merge, and Inspect the Dataset**

#### **1.3.1 Load and Inspect Dataset Structure [2 marks]**
**The Task:** Download the Olist dataset from Kaggle and load all **nine** relational CSV tables. Print the
shape and schema of each so you understand what you are working with.

**Hints & Tips:**
* The dataset slug is `olistbr/brazilian-ecommerce` (~100k orders across 9 tables).
* Key tables: `orders`, `order_items`, `order_reviews`, `order_payments`, `products`, `sellers`,
  `customers`, `geolocation`, `product_category_name_translation`.
* Note the join keys: `order_id`, `customer_id`, `product_id`, `seller_id`.
* Watch for the misspelled product columns: `product_name_lenght`, `product_description_lenght`.

**Why we are doing it:** You cannot merge tables correctly until you know each table's grain and keys.

**🧠 Learner Inference:** Each table answers a different question about an order. Your job in 1.3.2 is to
fold them all onto a single row per `order_id`.

In [3]:
# ---- Load the nine Olist tables ----
DATA_DIR = "."  # put the 9 CSVs in the same folder as this notebook (or point this at your folder / Drive path)

orders = pd.read_csv(f"{DATA_DIR}/olist_orders_dataset.csv")
order_items = pd.read_csv(f"{DATA_DIR}/olist_order_items_dataset.csv")
order_reviews = pd.read_csv(f"{DATA_DIR}/olist_order_reviews_dataset.csv")
order_payments = pd.read_csv(f"{DATA_DIR}/olist_order_payments_dataset.csv")
products = pd.read_csv(f"{DATA_DIR}/olist_products_dataset.csv")
sellers = pd.read_csv(f"{DATA_DIR}/olist_sellers_dataset.csv")
customers = pd.read_csv(f"{DATA_DIR}/olist_customers_dataset.csv")
geolocation = pd.read_csv(f"{DATA_DIR}/olist_geolocation_dataset.csv")
category_translation = pd.read_csv(f"{DATA_DIR}/product_category_name_translation.csv")

# Inspect the datasets
tables = {
    "orders": orders, "order_items": order_items, "order_reviews": order_reviews,
    "order_payments": order_payments, "products": products, "sellers": sellers,
    "customers": customers, "geolocation": geolocation,
    "category_translation": category_translation,
}
for name, df in tables.items():
    print(f"--- {name}: shape={df.shape} ---")
    print(df.dtypes)
    print()

--- orders: shape=(99441, 8) ---
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

--- order_items: shape=(112650, 7) ---
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

--- order_reviews: shape=(99224, 7) ---
review_id                    str
order_id                     str
review_score               int64
review_comment_title         str
review_comment_message       str
review_creation_date         str
review_answer_timestamp      str
dtype: object

--- order_payments: shape=(103886, 5) ---
order_id                    str
payment_sequential       

#### **1.3.2 Merge to an Order-Level Grain [3 marks]**
**The Task:** Collapse the nine tables into **one row per `order_id`**. Several tables have a finer grain
than the order (an order has many items and may have several payment records), so they must be
**aggregated before joining** — otherwise the merge multiplies rows and corrupts the target distribution.

**Hints & Tips:**
* **order_items → order level:** count items, **sum** `price` and `freight_value`. Pick a *primary item*
  (the most expensive line) to carry the representative `product_id` and `seller_id`.
* **order_payments → order level:** **sum** `payment_value`, take **max** `payment_installments`, count
  payment records, and take the **dominant** `payment_type` (highest-value record).
* **order_reviews → order level:** a few orders have multiple reviews — keep the **latest** one per order.
* Build the spine as `orders → customers`, then left-join the aggregated items/payments/reviews, then the
  primary product (+ English category) and seller.

**🔧 Design choice to document:** using the *most expensive* line as the primary item is a defensible rule
for multi-item orders.

**🧠 Learner Inference:** If your merged row count is wildly above ~100k, you forgot to aggregate a
one-to-many table before joining.

In [4]:
# ---- Aggregate order_items to order level ----
items_agg = order_items.groupby("order_id").agg(
    n_items=("order_item_id", "count"),
    price=("price", "sum"),
    freight_value=("freight_value", "sum"),
).reset_index()

# Primary item = most expensive line in the order (carries representative product & seller)
primary_idx = order_items.groupby("order_id")["price"].idxmax()
primary_item = order_items.loc[primary_idx, ["order_id", "product_id", "seller_id"]]
items_agg = items_agg.merge(primary_item, on="order_id", how="left")

# ---- Aggregate order_payments to order level ----
payments_agg = order_payments.groupby("order_id").agg(
    payment_value=("payment_value", "sum"),
    payment_installments=("payment_installments", "max"),
    n_payments=("payment_sequential", "count"),
).reset_index()

# Dominant payment type = type of the highest-value payment record
dom_idx = order_payments.groupby("order_id")["payment_value"].idxmax()
dominant_payment = order_payments.loc[dom_idx, ["order_id", "payment_type"]]
payments_agg = payments_agg.merge(dominant_payment, on="order_id", how="left")

print("items_agg:", items_agg.shape, "| dup order_id:", items_agg["order_id"].duplicated().sum())
print("payments_agg:", payments_agg.shape, "| dup order_id:", payments_agg["order_id"].duplicated().sum())

items_agg: (98666, 6) | dup order_id: 0
payments_agg: (99440, 5) | dup order_id: 0


In [5]:
# ---- Reduce reviews to one (latest) per order ----
order_reviews["review_creation_date"] = pd.to_datetime(order_reviews["review_creation_date"], errors="coerce")
order_reviews["review_answer_timestamp"] = pd.to_datetime(order_reviews["review_answer_timestamp"], errors="coerce")
reviews_sorted = order_reviews.sort_values(["order_id", "review_creation_date", "review_answer_timestamp"])
reviews_latest = reviews_sorted.drop_duplicates(subset="order_id", keep="last")
print("reviews_latest:", reviews_latest.shape, "| dup order_id:", reviews_latest["order_id"].duplicated().sum())

# ---- Build the order-level spine and join everything on order_id ----
spine = orders.merge(customers, on="customer_id", how="left")
spine = spine.merge(items_agg, on="order_id", how="left")
spine = spine.merge(payments_agg, on="order_id", how="left")
spine = spine.merge(reviews_latest, on="order_id", how="left")
print("spine after core joins:", spine.shape, "| dup order_id:", spine["order_id"].duplicated().sum())

reviews_latest: (98673, 7) | dup order_id: 0


spine after core joins: (99441, 27) | dup order_id: 0


In [6]:
# ---- Attach product attributes (+ English category) via the primary product_id ----
products_with_cat = products.merge(category_translation, on="product_category_name", how="left")
spine = spine.merge(products_with_cat, on="product_id", how="left")

# ---- Attach seller state via the primary seller_id ----
spine = spine.merge(sellers[["seller_id", "seller_state"]], on="seller_id", how="left")

print("spine after product/seller join:", spine.shape, "| dup order_id:", spine["order_id"].duplicated().sum())

spine after product/seller join: (99441, 37) | dup order_id: 0


#### **1.3.3 Profile Missing Values, Duplicates, and Ranges [1 mark]**
**The Task:** Profile the merged table: missing values per column, duplicate `order_id`s, and the numeric
ranges. This tells you what cleaning is required in Task 1.4.

**Hints & Tips:**
* `df.isnull().mean().sort_values(ascending=False)` gives the missing-fraction per column.
* Undelivered / cancelled orders are *expected* to have missing delivery timestamps — that's why we filter
  to delivered orders next.
* `df.describe()` reveals impossible values (e.g., negative freight) if any exist.

**🧠 Learner Inference:** High missingness in `order_delivered_customer_date` and `review_score` simply
reflects orders that were never delivered or never reviewed — they will be removed in 1.4.1.

In [7]:
# Data Profiling
print("Shape:", spine.shape)

print("\nMissing value fraction per column (top 15):")
print(spine.isnull().mean().sort_values(ascending=False).head(15))

print("\nDuplicate order_ids:", spine["order_id"].duplicated().sum())

print("\nNumeric ranges:")
print(spine.describe())

print("\norder_status breakdown:")
print(spine["order_status"].value_counts())

Shape: (99441, 37)

Missing value fraction per column (top 15):
review_comment_title             0.883851
review_comment_message           0.589978
order_delivered_customer_date    0.029817
product_category_name_english    0.022244
product_category_name            0.022023
product_photos_qty               0.022023
product_name_lenght              0.022023
product_description_lenght       0.022023
order_delivered_carrier_date     0.017930
product_width_cm                 0.007954
product_weight_g                 0.007954
product_height_cm                0.007954
product_length_cm                0.007954
product_id                       0.007794
freight_value                    0.007794
dtype: float64

Duplicate order_ids: 0

Numeric ranges:


       customer_zip_code_prefix       n_items         price  freight_value  payment_value  payment_installments    n_payments  review_score  \
count              99441.000000  98666.000000  98666.000000   98666.000000   99440.000000          99440.000000  99440.000000  98673.000000   
mean               35137.474583      1.141731    137.754076      22.823562     160.990267              2.930521      1.044710      4.086265   
min                 1003.000000      1.000000      0.850000       0.000000       0.000000              0.000000      1.000000      1.000000   
25%                11347.000000      1.000000     45.900000      13.850000      62.010000              1.000000      1.000000      4.000000   
50%                24416.000000      1.000000     86.900000      17.170000     105.290000              2.000000      1.000000      5.000000   
75%                58900.000000      1.000000    149.900000      24.040000     176.970000              4.000000      1.000000      5.000000   

### **Task 1.4: Clean and Prepare Analysis-Ready Data**

#### **1.4.1 Restrict to Delivered+Reviewed Orders & Remove Structural Issues [2 marks]**
**The Task:** Keep only orders that (a) were actually **delivered** and (b) have a **review score** — both
are required to build and validate the target. Then drop structural duplicates and rows missing the
essential fields.

**Hints & Tips:**
* Filter `order_status == "delivered"`.
* Require non-null `review_score` **and** non-null `order_delivered_customer_date`
  (a handful of "delivered" rows lack the actual delivery timestamp — drop those).
* Drop duplicate `order_id`s if any survived.
* Do **not** drop rows just because a *product attribute* is missing — leave those NaNs for Notebook 2 to
  handle (imputation is a post-split step).

**🧠 Learner Inference:** Restricting to delivered+reviewed is what defines our modelling population: orders
where a customer actually experienced fulfilment and told us how they felt.

In [8]:
# (a) delivered orders only
df = spine[spine["order_status"] == "delivered"].copy()
print("After delivered filter:", df.shape)

# (b) must have a review score
df = df[df["review_score"].notna()]
print("After review_score filter:", df.shape)

After delivered filter: (96478, 37)
After review_score filter: (95832, 37)


In [9]:
# (c) must have an actual delivery date
df = df[df["order_delivered_customer_date"].notna()]
print("After delivery-date filter:", df.shape)

# One delivered+reviewed order has ZERO matching payment records in this dataset -- a genuine
# data-quality gap, not something to invent with fillna. Payment facts are essential, so we drop it.
df = df[df["payment_type"].notna()]
print("After requiring a payment record:", df.shape)

# (d) drop structural duplicates
df = df.drop_duplicates()
df = df.drop_duplicates(subset="order_id", keep="first")
print("After dropping duplicates:", df.shape)

After delivery-date filter: (95824, 37)
After requiring a payment record: (95823, 37)


After dropping duplicates: (95823, 37)


#### **1.4.2 Standardise Schema, Categories, and Timestamps [3 marks]**
**The Task:** Convert all date columns to real datetimes, standardise text fields (lowercase/strip the
English category and payment type), and tidy dtypes so Notebook 2 can engineer features cleanly.

**Hints & Tips:**
* Parse every `*_date` / `*_timestamp` column with `pd.to_datetime(..., errors="coerce")`.
* Fill the English `product_category` for untranslated categories by falling back to the original
  Portuguese name (rather than leaving it blank).
* Strip/lowercase categorical strings so `"Credit_Card "` and `"credit_card"` don't become two categories.

**🧠 Learner Inference:** Standardising now means the delivery-delay and processing-time features you build
in Notebook 2 are just clean date subtractions.

In [10]:
# ---- Parse all date/time columns ----
date_cols = [
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date",
    "review_creation_date", "review_answer_timestamp",
]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# ---- Standardise categorical text ----
# Fall back to Portuguese name where English translation is missing
df["product_category_name_english"] = df["product_category_name_english"].fillna(df["product_category_name"])
df["product_category_name_english"] = df["product_category_name_english"].str.strip().str.lower()
df["payment_type"] = df["payment_type"].str.strip().str.lower()
df["customer_state"] = df["customer_state"].str.strip().str.upper()
df["seller_state"] = df["seller_state"].str.strip().str.upper()

# ---- Tidy a couple of dtypes ----
df["n_items"] = df["n_items"].astype(int)
df["n_payments"] = df["n_payments"].astype(int)
df["payment_installments"] = df["payment_installments"].astype(int)

print(df[["order_purchase_timestamp", "order_delivered_customer_date", "payment_type",
          "product_category_name_english", "n_items", "n_payments"]].dtypes)

order_purchase_timestamp         datetime64[us]
order_delivered_customer_date    datetime64[us]
payment_type                                str
product_category_name_english               str
n_items                                   int64
n_payments                                int64
dtype: object


#### **1.4.3 Create the Binary Target & Export the Leakage-Safe Dataset [2 marks]**
**The Task:** Create the binary classification target and export the analysis-ready, **split-ready** file
for Notebook 2.

**Hints & Tips:**
* `satisfied = (review_score >= 4).astype(int)` → 1 = satisfied (4–5★), 0 = unsatisfied (1–3★).
* Expect roughly **79% satisfied / 21% unsatisfied** — confirm this and note it.
* Keep the raw delivery timestamps and raw numeric/categorical columns; Notebook 2 will derive
  `delivery_delay_days`, `freight_ratio`, etc., **after** the split.

**⛔ Leakage guardrail:** Do **NOT** apply `StandardScaler`, mean/median imputation, target/frequency
encoding, or `seller_avg_score` here. Any statistic learned from the whole dataset would leak test
information into training. Those steps belong in Notebook 2, fit on the training split only.

**🧠 Learner Inference:** A clean, untransformed file is the contract with Notebook 2: it hands over *facts*
about each order, not *statistics* derived from the full population.

In [11]:
# ---- Binary target ----
df["satisfied"] = (df["review_score"] >= 4).astype(int)
print("Class balance:")
print(df["satisfied"].value_counts(normalize=True))

# ---- Select the columns we hand to Notebook 2 (raw facts, no derived statistics) ----
final_cols = [
    "order_id", "customer_id", "customer_unique_id", "customer_state", "customer_city",
    "order_status",
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date",
    "n_items", "price", "freight_value",
    "product_id", "product_category_name", "product_category_name_english",
    "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm",
    "product_photos_qty", "product_name_lenght", "product_description_lenght",
    "seller_id", "seller_state",
    "payment_type", "payment_value", "payment_installments", "n_payments",
    "review_score", "review_comment_title", "review_comment_message",
    "review_creation_date", "review_answer_timestamp",
    "satisfied",
]
df_final = df[final_cols].copy()
print("\nFinal shape:", df_final.shape)

Class balance:
satisfied
1    0.78933
0    0.21067
Name: proportion, dtype: float64

Final shape: (95823, 36)


---
## 💾 Save Artifact for Notebook 2

**IMPORTANT:** This cell writes `olist_clean.csv`. Notebook 2 depends on this file.

In [12]:
# Save the cleaned dataset
df_final.to_csv("olist_clean.csv", index=False)
print("Saved olist_clean.csv ->", df_final.shape)

Saved olist_clean.csv -> (95823, 36)


---
## ✅ END-OF-NOTEBOOK CHECKLIST

> **⚠️ Verify every item before moving to Notebook 2.**

- [ ] All **nine** Olist tables loaded and inspected (`.info()` / shapes printed)
- [ ] Tables merged to **one row per `order_id`** with documented grain collapse (items, payments, reviews)
- [ ] Missing values, duplicates, and numeric ranges profiled
- [ ] Restricted to **delivered + reviewed** orders; structural duplicates / absolute-missing rows removed
- [ ] Timestamps parsed; product categories translated to English; categorical text standardised
- [ ] Binary target `satisfied` created and class balance (~79/21) confirmed
- [ ] **NO** scaling, imputation, encoding, or `seller_avg_score` applied (leakage prevention)
- [ ] **`olist_clean.csv` saved to disk** ← _CRITICAL for Notebook 2_

**If any item is unchecked, fix it before proceeding.**